### Manual: 06-cuda-toolchain

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Pick a CUDA toolkit that can compile extensions for torch 1.10.1+cu113.

torch 1.10.1 refuses to build a CUDA extension when the host `nvcc` major
version differs from its own (11.3): see
`torch/utils/cpp_extension.py::BuildExtension._check_cuda_version`.
Kaggle's image ships CUDA 12.x only, and NVIDIA's ubuntu2404 apt repo has no
11.x packages, so the preferred route is the **ubuntu2204** CUDA repo (same
apt key, 11.8 packages, runs fine on the noble userland).

Order of preference (recorded in the artifact, never silent):
  1. an existing CUDA 11.x toolkit on disk;
  2. `cuda-nvcc-11-8` + `cuda-cudart-dev-11-8` from NVIDIA's ubuntu2204 repo;
  3. the system CUDA 12.x `nvcc` with torch's major-version guard bypassed --
     explicit, logged deviation, and the build either works or is reported.
"""

import glob
import json
import os
import re
import shutil
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

log = os.path.join(sc.LOGS, "cuda_toolchain.log")
KEYRING = "/usr/share/keyrings/cuda-archive-keyring.gpg"
SOURCE_LIST = "/etc/apt/sources.list.d/cuda-ubuntu2204.list"
REPO_2204 = "https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/"


def nvcc_release(nvcc):
    raw = sc.capture('"%s" --version' % nvcc)
    match = re.search(r"release (\d+\.\d+)", raw)
    return (match.group(1) if match else None), raw.splitlines()[-1] if raw else ""


def discover():
    found = []
    bases = []
    for line in sc.capture("ls -d /usr/local/cuda* 2>/dev/null").splitlines():
        bases.append(line.strip())
    for nvcc in glob.glob("/usr/local/cuda*/bin/nvcc"):
        bases.append(os.path.dirname(os.path.dirname(nvcc)))
    which = shutil.which("nvcc")
    if which:
        bases.append(os.path.dirname(os.path.dirname(which)))
    for base in sorted(set(b for b in bases if b)):
        nvcc = os.path.join(base, "bin", "nvcc")
        if not os.path.isfile(nvcc):
            continue
        release, raw = nvcc_release(nvcc)
        found.append(
            {
                "path": base,
                "nvcc": nvcc,
                "release": release,
                "raw": raw,
                "cudart_header": os.path.isfile(os.path.join(base, "include", "cuda_runtime.h")),
            }
        )
    return found


report = {"attempts": [], "toolkits_initial": discover()}
chosen = None
for toolkit in report["toolkits_initial"]:
    if (toolkit["release"] or "").startswith("11."):
        chosen = toolkit
        chosen["strategy"] = "existing-11x"
        break

# ------------------------------------------------------- apt route: CUDA 11.8
if chosen is None:
    attempt = {"strategy": "apt-ubuntu2204-cuda-11-8"}
    if not os.path.isfile(KEYRING):
        sc.sh(
            "curl -fsSL %s | gpg --dearmor -o %s" % (REPO_2204 + "3bf863cc.pub", KEYRING),
            log=log,
        )
    with open(SOURCE_LIST, "w") as handle:
        handle.write("deb [signed-by=%s] %s /\n" % (KEYRING, REPO_2204))
    sc.sh("apt-get update -qq", log=log, timeout=1200)
    attempt["apt_policy"] = sc.capture("apt-cache policy cuda-nvcc-11-8 | head -6")
    rc, _ = sc.sh(
        "apt-get install -y -qq --no-install-recommends cuda-nvcc-11-8 cuda-cudart-dev-11-8",
        log=log,
        timeout=3600,
    )
    attempt["install_rc"] = rc
    # CUDA 11.8 developer libraries live under lib*-dev-11-8 in NVIDIA's repo.
    # Installed separately so a rename in that repo cannot sink the toolkit.
    rc_libs, _ = sc.sh(
        "apt-get install -y -qq --no-install-recommends "
        "libcusparse-dev-11-8 libcublas-dev-11-8 libcurand-dev-11-8",
        log=log,
        timeout=3600,
    )
    attempt["install_libs_rc"] = rc_libs
    # cusolverDn.h is pulled in by ATen/cuda/CUDAContext.h (spconv .cc files).
    rc_solver, _ = sc.sh(
        "apt-get install -y -qq --no-install-recommends libcusolver-dev-11-8",
        log=log,
        timeout=3600,
    )
    attempt["install_cusolver_rc"] = rc_solver
    candidates = []
    for base in glob.glob("/usr/local/cuda-11*"):
        nvcc = os.path.join(base, "bin", "nvcc")
        if os.path.isfile(nvcc):
            release, raw = nvcc_release(nvcc)
            candidates.append(
                {
                    "path": base,
                    "nvcc": nvcc,
                    "release": release,
                    "raw": raw,
                    "cudart_header": os.path.isfile(
                        os.path.join(base, "include", "cuda_runtime.h")
                    ),
                }
            )
    attempt["toolkits_found"] = candidates
    report["attempts"].append(attempt)
    for toolkit in candidates:
        if (toolkit["release"] or "").startswith("11."):
            toolkit["strategy"] = "apt-ubuntu2204-cuda-11-8"
            chosen = toolkit
            break

# ------------------------------------------------- system nvcc + guard bypass
bypass_guard = False
if chosen is None:
    toolkits = discover()
    report["attempts"].append({"strategy": "system-nvcc-with-guard-bypass", "toolkits": toolkits})
    if toolkits:
        # Highest available toolkit, with the documented guard bypass.
        chosen = sorted(toolkits, key=lambda item: item["release"] or "0")[-1]
        chosen["strategy"] = "system-nvcc-with-guard-bypass"
        bypass_guard = True

# ------------------------------------------------------------ host compiler
# CUDA 11.8's nvcc rejects gcc-13 headers ("_Float32 is undefined"); the Kaggle
# image now ships gcc-13 only, so install gcc-11 when no 9/10/11 is present.
if not any(shutil.which(c) for c in ("gcc-11", "gcc-10", "gcc-9")):
    sc.sh(
        "apt-get install -y -qq --no-install-recommends gcc-11 g++-11",
        log=log,
        timeout=1800,
    )

gcc = None
gxx = None
for candidate in ("gcc-11", "gcc-10", "gcc-9", "gcc-12", "gcc-13"):
    path = shutil.which(candidate)
    if path:
        gcc = path
        gxx = shutil.which(candidate.replace("gcc", "g++"))
        break
report["gcc_default"] = sc.capture("gcc --version 2>/dev/null | head -1")
report["gcc_chosen"] = gcc or "system default"

report["chosen"] = chosen
report["bypass_cuda_guard"] = bypass_guard
report["apt_source_list"] = sc.capture("cat %s 2>/dev/null" % SOURCE_LIST)
report["apt_keyring_present"] = os.path.isfile(KEYRING)
report["ok"] = chosen is not None

if chosen is not None:
    sc.update(
        build_toolchain={
            "cuda_home": chosen["path"],
            "cuda_release": chosen["release"],
            "strategy": chosen["strategy"],
            "bypass_cuda_guard": bypass_guard,
            "cc": gcc,
            "cxx": gxx,
            "cudart_header": chosen.get("cudart_header"),
        }
    )
sc.save_json("cuda_toolchain.json", report)
print(json.dumps(report, indent=2, sort_keys=True))
print("TOOLCHAIN_OK" if report["ok"] else "TOOLCHAIN_FAILED")
